# Capítulo 1 — Mocks y aislamiento en testing

# Introducción

Uno de los errores más comunes al aprender testing es pensar que:

> “hacer tests” significa solamente escribir `assert`.

Pero en proyectos reales aparece un problema mucho más difícil:

```text
¿Cómo probamos código que depende del mundo externo?
```

Por ejemplo:

- APIs,
- internet,
- bases de datos,
- archivos,
- tiempo,
- correos,
- servicios de terceros.

Ese es el verdadero origen de los mocks.

---

# Objetivos del capítulo

Al terminar este notebook deberías poder:

- entender qué es una dependencia,
- reconocer tests frágiles,
- usar mocks correctamente,
- evitar falsos positivos,
- y diseñar tests aislados y determinísticos.


# ¿Qué es una dependencia?

Una dependencia es cualquier cosa que nuestro código necesita para funcionar
pero que NO controlamos completamente.

## Ejemplos

### Dependencias externas

- internet,
- APIs,
- servidores,
- sistemas operativos,
- bases de datos.

### Dependencias temporales

- hora actual,
- fechas,
- timers.

### Dependencias físicas

- archivos,
- discos,
- sensores,
- hardware.

---

# Problema

Las dependencias hacen que los tests sean:

- lentos,
- difíciles de reproducir,
- impredecibles,
- costosos.


In [ ]:
import requests

def obtener_usuario(user_id):
    response = requests.get(
        f'https://jsonplaceholder.typicode.com/users/{user_id}'
    )

    return response.json()

# Primer análisis

Este código parece completamente normal.

Pero pensemos como testers.

## Preguntas importantes

- ¿Qué ocurre si no hay internet?
- ¿Qué pasa si la API responde lento?
- ¿Qué ocurre si la API cambia?
- ¿Qué pasa si el servidor devuelve 500?
- ¿Qué ocurre si devuelve JSON inválido?

Aquí aparece la idea de fragilidad.


In [ ]:
usuario = obtener_usuario(1)

print(usuario["name"])

# El problema de los tests reales

Imagina este test:

```python
assert obtener_usuario(1)["id"] == 1
```

Parece correcto.

Pero realmente depende de:

- internet,
- DNS,
- latencia,
- disponibilidad del servidor,
- estabilidad del endpoint.

Eso significa que el test puede fallar
aunque nuestro código esté perfectamente bien.


# Aislamiento

Uno de los principios más importantes del testing moderno es:

> Los tests deberían aislar el sistema bajo prueba.

## Modelo mental

```text
Sistema bajo prueba
│
├── lógica propia
└── dependencias externas
```

La idea es:

- probar nuestra lógica,
- NO probar internet.


# ¿Qué es un mock?

Un mock es un objeto falso que reemplaza una dependencia real.

Su objetivo NO es “hacer que todo pase”.

Su objetivo es:

```text
controlar comportamiento externo
```

---

# Analogía

Imagina entrenar pilotos.

No usamos un avión real para cada práctica.

Usamos simuladores.

Los mocks son simuladores para software.


In [ ]:
from unittest.mock import patch, Mock

fake_response = Mock()
fake_response.json.return_value = {
    "id": 1,
    "name": "Ada Lovelace"
}

with patch("requests.get", return_value=fake_response):
    usuario = obtener_usuario(1)

print(usuario)

# ¿Qué acaba de ocurrir?

`requests.get` fue reemplazado temporalmente.

## Antes

```text
Test → Internet → API real
```

## Después

```text
Test → Mock controlado
```

Ahora:

- no usamos internet,
- el resultado es instantáneo,
- el comportamiento es reproducible.


# Ventajas de los mocks

## Velocidad

Los tests son mucho más rápidos.

## Reproducibilidad

Siempre producen el mismo resultado.

## Control

Podemos simular errores difíciles de provocar.

## Observabilidad

Podemos medir llamadas y argumentos.


In [ ]:
from requests.exceptions import Timeout

with patch("requests.get", side_effect=Timeout):
    try:
        obtener_usuario(1)
    except Exception as e:
        print(type(e))

# side_effect

`side_effect` es extremadamente poderoso.

Permite:

- lanzar excepciones,
- devolver múltiples valores,
- simular secuencias temporales.

Esto es clave para probar resiliencia.


In [ ]:
ok = Mock()
ok.json.return_value = {"ok": True}

with patch(
    "requests.get",
    side_effect=[Timeout(), Timeout(), ok]
):
    print(obtener_usuario(1))

# Simulación temporal

Aquí simulamos:

```text
Intento 1 → timeout
Intento 2 → timeout
Intento 3 → éxito
```

Esto sería muy difícil de reproducir en sistemas reales.


# Verificando comportamiento

Los mocks no solo devuelven datos.

También registran información sobre uso.


In [ ]:
with patch("requests.get", return_value=fake_response) as mock_get:
    obtener_usuario(1)

    print(mock_get.call_count)
    print(mock_get.called)

# Problema: mocks mal usados

Un error muy común:

```python
with patch("requests.get"):
    pass
```

Eso produce tests vacíos.

---

# Riesgo

Un mock mal diseñado puede:

- ocultar bugs reales,
- producir falsos positivos,
- generar falsa confianza.


# Buenas prácticas

## Usa mocks solo donde tenga sentido

No mockees toda la aplicación.

## Mockea fronteras externas

- APIs,
- DBs,
- archivos,
- tiempo.

## No mockees lógica propia innecesariamente

Eso destruye valor del test.

## Verifica comportamiento importante

No solo que “algo fue llamado”.


# Reflexión final

Los mocks existen porque el mundo real es impredecible.

El objetivo del testing no es:

```text
hacer que todo pase
```

El objetivo es:

```text
detectar comportamiento incorrecto
```
